# Fase 4 — Consolidación del Proyecto, Visualización Explicativa y Comunicación de Resultados (SIDREP 2023–2025)
**Curso:** MCDI500 — Programación para la Ciencia de Datos (Grupo 3)  
**Docente:** Dr. Omar Iván Salinas Silva  
**Integrantes (composición activa del Grupo 3):** Victoria Álvarez Pinto, Ignacio Peña Pizarro y Raúl Hormazábal Figueroa

---

## Integración de las Cuatro Fases del Proyecto (`F1` → `F2` → `F3` → `F4`)
Este cuaderno cierra el flujo técnico del proyecto partiendo **exclusivamente de los cuatro artefactos exportados por el `Pipeline` orientado a objetos de la Fase 3** (`entrenamiento_f3.csv`, `prueba_f3.csv`, `diccionario_f3.csv` y `parametros_f3.csv`), sin volver a limpiar ni a reprocesar la base bruta:

| Fase | Foco Técnico | Artefacto de Salida que Conecta con la Fase Siguiente |
| :--- | :--- | :--- |
| **Fase 1** (`F1_Definicion.ipynb`) | Pregunta analítica, 3 objetivos específicos (`OE1`–`OE3`), alcance/restricciones/supuestos separados e ingesta de `Base_Maestra_2023_2025.xlsx` (`SHA-256` verificado). | Diagnóstico estructural (`118.280 × 21`) y contrato de variables. |
| **Fase 2** (`F2_Preprocesamiento.ipynb`) | Limpieza determinista (`-118` duplicados, `-21` nulos en `cantidad_kg` sin imputar variable objetivo, rescate vectorizado de `39.381` coordenadas de 2024 y normalización $O(U \cdot L + N)$ de `812` glosas). | Dataset depurado (`118.141` declaraciones válidas de `12.189` establecimientos únicos `id_vu`). |
| **Fase 3** (`F3_NucleoAlgoritmico_POO.ipynb`) | Jerarquía `Transformador(ABC)`, `deepcopy`, `SHA-256` (`6.777` RUT), patrones `Strategy` y `Observer` doble, equivalencia exacta con versión funcional y suite `pytest`. | Exportación formal de `entrenamiento_f3.csv`, `prueba_f3.csv`, `diccionario_f3.csv` y `parametros_f3.csv`. |
| **Fase 4** (`F4_Visualizacion_Comunicacion.ipynb`) | Copia interpretable, unidad de análisis independiente por establecimiento (`N = 12.189`), descomposición interanual exacta (`Persistentes + Entrantes - Salientes`), curva de Lorenz/Gini y 4 figuras explicativas con IC de Wilson al 95 %. | `F4/figuras/fig1..fig4.png` (`200 dpi`), `F4/docs/hallazgos_f4.csv` (con `ic_wilson_95` y `celda_notebook`) y `F4/docs/metadatos_f4.json`. |


In [ ]:
# Celda In[1]: Verificación de entorno virtual, importación de src/comunicacion.py y paleta preatentiva
from pathlib import Path
import json
import sys
import timeit
import tracemalloc
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

RAIZ_PROYECTO = Path.cwd().resolve()
while not (RAIZ_PROYECTO / "src").exists() and RAIZ_PROYECTO != RAIZ_PROYECTO.parent:
    RAIZ_PROYECTO = RAIZ_PROYECTO.parent
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

from src.medicion import obtener_info_entorno
from src.comunicacion import (
    ARTEFACTOS_F3,
    SHA256_BASE_MAESTRA_OFICIAL,
    fmt_miles,
    fmt_dec,
    fmt_pct,
    verificar_sha256_origen,
    cargar_artefactos,
    leer_parametros,
    revertir_escalamiento,
    reconstruir_categorias,
    copia_interpretable,
    resumir_por_establecimiento,
    descomponer_cambio_interanual,
    calcular_concentracion_y_lorenz,
    auditar_rotacion_administrativa,
    intervalo_wilson,
    tasa_por_grupo,
    guardar_figura,
)

EN_VENV = sys.prefix != getattr(sys, "base_prefix", sys.prefix)
info_entorno = obtener_info_entorno()
print(f"[OK] Entorno virtual aislado activo: {EN_VENV} | Python {info_entorno['python_version']} | SO: {info_entorno['sistema_operativo']}")
print(f"[OK] Versiones fijadas: pandas=={pd.__version__}, numpy=={np.__version__}, matplotlib=={mpl.__version__}, seaborn=={sns.__version__}")

# Paleta institucional con contraste figura-fondo y jerarquía tipográfica
GRANATE = "#7A1C1C"   # Foco preatentivo: evidencia principal / IC 95% sobre el promedio nacional
AZUL    = "#1F4E79"   # Comparación secundaria / IC 95% que incluye al promedio nacional
GRIS    = "#6E7B85"   # Contexto o referencia / IC 95% bajo el promedio nacional
HIELO   = "#D7E3EE"   # Fondo suave de anotaciones
DORADO  = "#C58B00"   # Marcador de contraste sin Minería

mpl.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#CCCCCC",
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.titlesize": 11.5,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "font.sans-serif": ["DejaVu Sans", "Arial", "Calibri"],
    "grid.color": "#E6ECF2",
    "grid.linestyle": "--",
    "grid.linewidth": 0.6,
})


## 1. Carga y Verificación del Contrato de los Cuatro Artefactos de la Fase 3
Siguiendo el principio de no reprocesamiento en la Fase 4, cargamos únicamente los cuatro artefactos exportados por el `Pipeline` de la Fase 3 (`entrenamiento_f3.csv`, `prueba_f3.csv`, `diccionario_f3.csv` y `parametros_f3.csv`) y verificamos su contrato estructural con cinco aserciones explícitas (`assert`).


In [ ]:
# Celda In[2]: Carga de los 4 artefactos de Fase 3 y verificación del contrato estructural
CARPETA_PROCESADOS = RAIZ_PROYECTO / "F2" / "data" / "processed"
artefactos = cargar_artefactos(CARPETA_PROCESADOS)
parametros = leer_parametros(artefactos["parametros"])

entrenamiento = artefactos["entrenamiento"]
prueba = artefactos["prueba"]
diccionario = artefactos["diccionario"]

# 5 verificaciones por contrato sobre los artefactos recibidos de F3
assert list(entrenamiento.columns) == list(prueba.columns), "Las columnas de entrenamiento y prueba no coinciden."
assert set(entrenamiento.columns) == set(diccionario["columna"]), "El diccionario no coincide con las columnas de la matriz."
assert int(entrenamiento.isna().sum().sum()) == 0, "Existen valores faltantes en entrenamiento_f3.csv."
assert int(prueba.isna().sum().sum()) == 0, "Existen valores faltantes en prueba_f3.csv."
assert {"ImputadorPorGrupo", "CodificadorCategorias", "EscaladorRobusto"} <= set(parametros.keys()), "Faltan pasos en parametros_f3.csv."

resumen_artefactos = pd.DataFrame([
    {"Artefacto": "entrenamiento_f3.csv", "Filas": fmt_miles(len(entrenamiento)), "Columnas": len(entrenamiento.columns), "Nulos": int(entrenamiento.isna().sum().sum())},
    {"Artefacto": "prueba_f3.csv", "Filas": fmt_miles(len(prueba)), "Columnas": len(prueba.columns), "Nulos": int(prueba.isna().sum().sum())},
    {"Artefacto": "diccionario_f3.csv", "Filas": fmt_miles(len(diccionario)), "Columnas": len(diccionario.columns), "Nulos": int(diccionario.isna().sum().sum())},
    {"Artefacto": "parametros_f3.csv", "Filas": fmt_miles(len(artefactos["parametros"])), "Columnas": len(artefactos["parametros"].columns), "Nulos": int(artefactos["parametros"].isna().sum().sum())},
])
display(resumen_artefactos)


## 2. Construcción de la Copia Interpretable, Verificación SHA-256 y Unidad de Análisis por Establecimiento
El conjunto escalado con $z$-scores robustos y codificado en `0/1` sirve para modelar, pero no para comunicar a un tomador de decisiones. Mediante `copia_interpretable(df, parametros)` invertimos matemáticamente el escalamiento robusto ($x = x_{\text{rob}} \cdot \text{RIC} + \text{mediana}$) y reconstruimos las regiones, estados físicos, comunas y rubros desde los parámetros guardados en `parametros_f3.csv`.

Además, verificamos la huella criptográfica **SHA-256 (`7de8ff0ed46488dcc52a551788d12bc3e9c6599a7b9afd1c30eb1139c60dc44b`)** del archivo de origen `Base_Maestra_2023_2025.xlsx` antes de contrastar con `np.allclose` la exactitud de las toneladas recuperadas. Finalmente, mediante `resumir_por_establecimiento(legible)` consolidamos las `118.141` declaraciones en las **`12.189` instalaciones físicas únicas (`id_vu`)**, que constituyen observaciones independientes para el cálculo de intervalos de confianza de Wilson al 95 % en las Figuras 3 y 4.


In [ ]:
# Celda In[3]: Copia interpretable, verificación SHA-256 del Excel original, unidad por establecimiento y benchmark
procesado_completo = pd.concat([entrenamiento, prueba], ignore_index=True)
legible = copia_interpretable(procesado_completo, parametros)

# 1. Verificación criptográfica SHA-256 y contraste independiente contra el Excel de origen
RUTA_EXCEL_ORIGEN = RAIZ_PROYECTO / "F1" / "data" / "raw" / "Base_Maestra_2023_2025.xlsx"
sha256_verificado = verificar_sha256_origen(RUTA_EXCEL_ORIGEN, SHA256_BASE_MAESTRA_OFICIAL)
print(f"[OK] Huella SHA-256 verificada en Base_Maestra_2023_2025.xlsx: {sha256_verificado}")

assert np.allclose(legible["cantidad_kg_revertida"], legible["cantidad_kg"], rtol=1e-6, atol=1e-5), "Error en reversión de cantidad_kg"
assert np.allclose(legible["latitud_revertida"], legible["latitud"], rtol=1e-6, atol=1e-5), "Error en reversión de latitud"
dif_max_kg = float(np.max(np.abs(legible["cantidad_kg_revertida"] - legible["cantidad_kg"])))
print(f"[OK] Verificación np.allclose exitosa sobre {fmt_miles(len(legible))} declaraciones ({fmt_dec(legible['cantidad_ton'].sum(), 1)} t) | Diferencia máx.: {dif_max_kg:.2e} kg")

# 2. Consolidación por establecimiento único (N = 12.189 independientes) y por establecimiento-año (N = 28.099)
df_est, df_est_anio = resumir_por_establecimiento(legible, umbral_ton_anual=12.0)
print(
    f"[OK] Unidades de análisis construidas: {fmt_miles(len(df_est))} establecimientos únicos (id_vu) | "
    f"{fmt_miles(len(df_est_anio))} pares establecimiento-año | {fmt_miles(len(legible))} declaraciones."
)

# 3. Medición empírica de eficiencia (timeit y tracemalloc): reversión por filas vs vectorizada
def _revertir_por_comprension(df_in, params_in):
    out = df_in.copy()
    for col, info in params_in["EscaladorRobusto"].items():
        med = float(info["mediana"])
        ric = float(info["ric"])
        out[f"{col}_revertida"] = [v * ric + med for v in out[f"{col}_rob"]]
    return out

tracemalloc.start()
t_filas = min(timeit.repeat(lambda: _revertir_por_comprension(entrenamiento, parametros), repeat=3, number=1))
_, mem_filas = tracemalloc.get_traced_memory()
tracemalloc.stop()

tracemalloc.start()
t_vec = min(timeit.repeat(lambda: revertir_escalamiento(entrenamiento, parametros), repeat=3, number=1))
_, mem_vec = tracemalloc.get_traced_memory()
tracemalloc.stop()

aceleracion_f4 = t_filas / max(t_vec, 1e-9)
print(
    f"[OK] Eficiencia reversión (N={fmt_miles(len(entrenamiento))}, 5 vars): "
    f"Comprensión={t_filas:.5f} s ({mem_filas/(1024**2):.2f} MB) vs "
    f"Vectorizada={t_vec:.5f} s ({mem_vec/(1024**2):.2f} MB) -> {fmt_dec(aceleracion_f4, 1)}x más rápida"
)


# Celda In[3]: Copia interpretable, verificación SHA-256 del Excel original, unidad por establecimiento y benchmark
procesado_completo = pd.concat([entrenamiento, prueba], ignore_index=True)
legible = copia_interpretable(procesado_completo, parametros)

# 1. Verificación criptográfica SHA-256 y contraste independiente contra el Excel de origen
RUTA_EXCEL_ORIGEN = RAIZ_PROYECTO / "F1" / "data" / "raw" / "Base_Maestra_2023_2025.xlsx"
sha256_verificado = verificar_sha256_origen(RUTA_EXCEL_ORIGEN, SHA256_BASE_MAESTRA_OFICIAL)
print(f"[OK] Huella SHA-256 verificada en Base_Maestra_2023_2025.xlsx: {sha256_verificado}")

assert np.allclose(legible["cantidad_kg_revertida"], legible["cantidad_kg"], rtol=1e-6, atol=1e-5), "Error en reversión de cantidad_kg"
assert np.allclose(legible["latitud_revertida"], legible["latitud"], rtol=1e-6, atol=1e-5), "Error en reversión de latitud"
dif_max_kg = float(np.max(np.abs(legible["cantidad_kg_revertida"] - legible["cantidad_kg"])))
print(f"[OK] Verificación np.allclose exitosa sobre {fmt_miles(len(legible))} declaraciones ({fmt_dec(legible['cantidad_ton'].sum(), 1)} t) | Diferencia máx.: {dif_max_kg:.2e} kg")

# 2. Consolidación por establecimiento único (N = 12.189 independientes) y por establecimiento-año (N = 28.099)
df_est, df_est_anio = resumir_por_establecimiento(legible, umbral_ton_anual=12.0)
print(
    f"[OK] Unidades de análisis construidas: {fmt_miles(len(df_est))} establecimientos únicos (id_vu) | "
    f"{fmt_miles(len(df_est_anio))} pares establecimiento-año | {fmt_miles(len(legible))} declaraciones."
)

# 3. Medición empírica de eficiencia (timeit y tracemalloc): reversión por filas vs vectorizada
def _revertir_por_comprension(df_in, params_in):
    out = df_in.copy()
    for col, info in params_in["EscaladorRobusto"].items():
        med = float(info["mediana"])
        ric = float(info["ric"])
        out[f"{col}_revertida"] = [v * ric + med for v in out[f"{col}_rob"]]
    return out

tracemalloc.start()
t_filas = min(timeit.repeat(lambda: _revertir_por_comprension(entrenamiento, parametros), repeat=3, number=1))
_, mem_filas = tracemalloc.get_traced_memory()
tracemalloc.stop()

tracemalloc.start()
t_vec = min(timeit.repeat(lambda: revertir_escalamiento(entrenamiento, parametros), repeat=3, number=1))
_, mem_vec = tracemalloc.get_traced_memory()
tracemalloc.stop()

aceleracion_f4 = t_filas / max(t_vec, 1e-9)
print(
    f"[OK] Eficiencia reversión (N={fmt_miles(len(entrenamiento))}, 5 vars): "
    f"Comprensión={t_filas:.5f} s ({mem_filas/(1024**2):.2f} MB) vs "
    f"Vectorizada={t_vec:.5f} s ({mem_vec/(1024**2):.2f} MB) -> {fmt_dec(aceleracion_f4, 1)}x más rápida"
)


In [ ]:
# Celda In[4]: Cálculos de concentración (Lorenz/Gini), descomposición interanual, sensibilidad, rotación y tabla de objetivos
conc_total = calcular_concentracion_y_lorenz(df_est["ton_total_trienio"])
conc_nucleo = calcular_concentracion_y_lorenz(df_est.loc[df_est["es_nucleo_persistente"] == 1, "ton_total_trienio"])

# Descomposición interanual exacta (padrón completo y sensibilidad sin Top 1 planta y sin Top 1% = 122 plantas)
ranking_est = df_est.sort_values("ton_total_trienio", ascending=False)
id_top1_planta = [int(ranking_est.iloc[0]["id_vu"])]
ids_top1_pct = [int(x) for x in ranking_est.iloc[:conc_total["n_top1_pct"]]["id_vu"]]

desc_23_24 = descomponer_cambio_interanual(legible, 2023, 2024)
desc_24_25 = descomponer_cambio_interanual(legible, 2024, 2025)
desc_23_25 = descomponer_cambio_interanual(legible, 2023, 2025)
desc_23_25_sin_top1 = descomponer_cambio_interanual(legible, 2023, 2025, excluir_ids=id_top1_planta)
desc_23_25_sin_top1pct = descomponer_cambio_interanual(legible, 2023, 2025, excluir_ids=ids_top1_pct)

tabla_descomposicion = pd.DataFrame([
    {"Escenario / Par de años": "Padrón completo (2023 → 2024)", **desc_23_24},
    {"Escenario / Par de años": "Padrón completo (2024 → 2025)", **desc_24_25},
    {"Escenario / Par de años": "Padrón completo (2023 → 2025)", **desc_23_25},
    {"Escenario / Par de años": f"Sensibilidad sin Top 1 planta id_vu={id_top1_planta[0]} (2023 → 2025)", **desc_23_25_sin_top1},
    {"Escenario / Par de años": f"Sensibilidad sin Top 1 % = {len(ids_top1_pct)} plantas (2023 → 2025)", **desc_23_25_sin_top1pct},
])
print("=== 1. DESCOMPOSICIÓN EXACTA DEL CAMBIO INTERANUAL Y SENSIBILIDAD (toneladas) ===")
display(tabla_descomposicion[[
    "Escenario / Par de años", "vol_inicio_ton", "vol_fin_ton", "cambio_total_ton",
    "variacion_persistentes_ton", "volumen_entrantes_ton", "volumen_salientes_ton",
    "n_persistentes", "n_entrantes", "n_salientes"
]])

# Auditoría de rotación administrativa por coincidencia en (comuna, frec_RUT, rubro)
rot_23_24 = auditar_rotacion_administrativa(legible, 2023, 2024)
rot_24_25 = auditar_rotacion_administrativa(legible, 2024, 2025)
print("\n=== 2. AUDITORÍA DE ROTACIÓN ADMINISTRATIVA (COINCIDENCIA EN COMUNA + FREC_RUT + RUBRO) ===")
display(pd.DataFrame([rot_23_24, rot_24_25]))

# Tabla de vinculación entre los 3 Objetivos Específicos de F1 y las 4 Figuras de F4
tabla_vinculo_oe = pd.DataFrame([
    {
        "Objetivo Específico (F1)": "OE1 (Persistencia y concentración extrema de masa)",
        "Movimiento Narrativo": "1. Contexto estructural",
        "Figura en F4": "Figura 1 (Paneles A y B)",
        "Hallazgo Cuantitativo Principal": (
            f"El núcleo persistente ({fmt_pct(55.66)} de plantas, {fmt_miles(6785)} instalaciones) concentra el "
            f"{fmt_pct(95.67)} del volumen ({fmt_dec(1857961.4, 1)} t); dentro de él, su 1 % superior ({conc_nucleo['n_top1_pct']} plantas) "
            f"concentra el {fmt_pct(conc_nucleo['share_top1_pct'])} y el 10 % superior ({conc_nucleo['n_top10_pct']} plantas) el {fmt_pct(conc_nucleo['share_top10_pct'])} (Gini = {fmt_dec(conc_nucleo['gini'], 4)})."
        ),
    },
    {
        "Objetivo Específico (F1)": "OE2 (Descomposición interanual y concentración por rubro)",
        "Movimiento Narrativo": "2. Contraste temporal y sectorial",
        "Figura en F4": "Figura 2 (Paneles A y B)",
        "Hallazgo Cuantitativo Principal": (
            f"El núcleo persistente creció +{fmt_dec(30.8552, 1)} mil t entre 2023 y 2025, explicando todo el crecimiento neto nacional "
            f"(+{fmt_dec(28.7093, 1)} mil t, +{fmt_pct(4.5)}) y compensando la baja neta (-{fmt_dec(2.1459, 1)} mil t) de los no persistentes. "
            f"Minería aporta el {fmt_pct(41.25)} del volumen con solo {fmt_miles(4102)} declaraciones ({fmt_pct(3.47)} de los registros)."
        ),
    },
    {
        "Objetivo Específico (F1)": "OE3 (Gradiente regional por establecimiento con IC Wilson 95 %)",
        "Movimiento Narrativo": "3. Resolución territorial",
        "Figura en F4": "Figura 3 (Paneles A y B)",
        "Hallazgo Cuantitativo Principal": (
            "Por establecimiento único independiente (N = 12.189), Antofagasta (48,0 %, IC [43,8 %; 52,3 %]) y RM (32,7 %, IC [31,2 %; 34,2 %]) "
            "quedan con su IC 95 % completamente sobre el promedio nacional de Gran Generador (24,2 %), seguidas por Arica (27,5 %), Atacama (26,9 %) y Tarapacá (24,4 %)."
        ),
    },
    {
        "Objetivo Específico (F1)": "OE3 (Doble control de confusión: Minería y Macrozona Norte)",
        "Movimiento Narrativo": "4. Discusión y control estratificado",
        "Figura en F4": "Figura 4 (Paneles A y B)",
        "Hallazgo Cuantitativo Principal": (
            "La asociación entre persistencia trienal y Gran Generador se mantiene al controlar por Minería (73,5 % vs 19,8 % en Minería; "
            "35,1 % vs 9,0 % en otros 18 rubros) y al agregar la Macrozona Norte como segundo control (40,4 % vs 11,7 % en rubros no mineros del Norte y 34,3 % vs 8,6 % en el Resto del país)."
        ),
    },
])
print("\n=== 3. VINCULACIÓN ENTRE OBJETIVOS ESPECÍFICOS (F1) Y FIGURAS EXPLICATIVAS (F4) ===")
display(tabla_vinculo_oe)


## 4. Figuras 1 y 2: Concentración en el Núcleo Persistente (Curva de Lorenz) y Descomposición Interanual / Sectorial (`OE1` y `OE2`)

### Lectura en 4 enunciados de la Figura 1 (`OE1` — Contexto y concentración interna)
1. **Qué muestra:** El Panel A contrasta la distribución de los `12.189` establecimientos únicos (`id_vu`) frente a su participación en los `1.942.051,2 t` declarados según trayectoria (`1`, `2` o `3` años); el Panel B presenta la **Curva de Lorenz** y la participación acumulada del **1 %** y **10 % superior** de los establecimientos tanto dentro del **núcleo persistente (`6.785` plantas)** como en el padrón total.
2. **Qué se infiere:** Aunque el `44,3 %` de los establecimientos (`5.404` plantas ocasionales o intermitentes) entra o sale del padrón, entre todos ellos apenas suman el `4,3 %` de la masa nacional (`84.089,8 t`). El `55,7 %` de las plantas que conforman el **núcleo persistente (`6.785` instalaciones)** concentra el **`95,7 %` del tonelaje nacional (`1.857.961,4 t`)**, y dentro de ese mismo núcleo persistente existe una hiperconcentración extrema ($	ext{Gini} = 0,9421$, mediana trienal por planta `11,7 t`, `RIC [2,6; 47,6] t`): **el `1 %` superior (`68` plantas) concentra el `71,2 %` del volumen del núcleo (`1.322,9 mil t`) y el `10 %` superior (`678` plantas) concentra el `92,0 %` (`1.708,8 mil t`)**.
3. **Qué límite tiene:** La clasificación de persistencia se basa en la continuidad del identificador `id_vu` en una ventana observacional de tres años (2023–2025).
4. **Cómo aporta al relato:** Demuestra que la masa crítica de residuos peligrosos no solo se concentra en el núcleo persistente, sino específicamente en su decil y percentil superior de grandes complejos industriales.

### Lectura en 4 enunciados de la Figura 2 (`OE2` — Descomposición interanual y desproporción sectorial)
1. **Qué muestra:** El Panel A presenta la evolución interanual 2023–2025 (`639,4 mil t → 634,5 mil t → 668,1 mil t`) y la descomposición exacta del cambio neto (`+28,7 mil t`) entre el crecimiento del núcleo persistente (`+30,9 mil t`) y la disminución neta de los grupos no persistentes (`-2,1 mil t`), junto con los flujos par a par de persistentes, entrantes y salientes; el Panel B muestra los 8 principales rubros por volumen y desagrega la composición interna de **`Otras actividades` (`41.589` declaraciones)** según su clasificador CIIU4.
2. **Qué se infiere:** El núcleo persistente aumentó en **`+30.855,2 t` (`606,3 mil` a `637,1 mil t`)**, lo que **explica todo el crecimiento neto nacional (`+28.709,3 t`, `+4,5 %`) y compensa la caída neta conjunta (`-2.145,9 t`) de los grupos intermitentes (`-12.032,0 t`) y ocasionales (`+9.886,1 t`)**. A nivel sectorial, **Minería aporta el `41,3 %` de toda la masa nacional (`801,1 mil t`) con apenas `4.102` declaraciones (`3,5 %` del total de registros)**, mientras que la categoría administrativa **`Otras actividades` (`294,6 mil t`, `15,2 %` en `41.589` declaraciones)** agrupa principalmente **logística, almacenamiento y transporte ferroviario/carretero (`151,1 mil t`)**, **hospitales públicos y privados (`35,3 mil t` en `12.793` declaraciones)** y **servicios de descontaminación y gas (`21,8 mil t`)**.
3. **Qué límite tiene:** El análisis de sensibilidad revela que el mayor establecimiento minero (`id_vu = 5440993`, en Antofagasta) creció `+45.251,4 t` entre 2023 y 2025; sin ese único generador la variación nacional pasa a `-16.542,1 t`, y al excluir el 1 % superior (`122` plantas) el resto del país (`12.067` plantas) presenta un crecimiento moderado de `+10.246,6 t`.
4. **Cómo aporta al relato:** Conecta la concentración temporal con la estructura productiva, justificando por qué es indispensable controlar por Minería y por región al analizar la distribución territorial.


In [ ]:
# Celda In[5]: Generación de Figura 1 (Persistencia y Curva de Lorenz del núcleo) y Figura 2 (Evolución, descomposición y rubros)
# --- FIGURA 1: Persistencia vs Masa (Panel A) y Curva de Lorenz / Top 1% y 10% en el Núcleo Persistente (Panel B) ---
orden_tray = [
    "Ocasional: entra o sale (1 año)",
    "Intermitente (2 años)",
    "Núcleo persistente (3 años)",
]
etiquetas_tray = [
    "Ocasional\n(1 año)",
    "Intermitente\n(2 años)",
    "Núcleo persistente\n(3 años: 2023–2025)",
]

plantas_por_g = df_est.groupby("trayectoria_vu")["id_vu"].count().reindex(orden_tray)
ton_por_g = df_est.groupby("trayectoria_vu")["ton_total_trienio"].sum().reindex(orden_tray)
pct_plantas = (plantas_por_g / plantas_por_g.sum()) * 100.0
pct_ton = (ton_por_g / ton_por_g.sum()) * 100.0

fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.6, 5.4), gridspec_kw={"width_ratios": [1.08, 1.12]})
fig1.suptitle(
    "El núcleo persistente (55,7 % de plantas) aporta el 95,7 % del volumen, y su 1 % superior (68 plantas) concentra el 71,2 %",
    fontsize=12.5, fontweight="bold", y=0.99, color="#06304A"
)

x_pos = np.arange(len(orden_tray))
ancho = 0.36
b_pl = ax1.bar(x_pos - ancho/2, pct_plantas.values, width=ancho, color=AZUL, label="% de establecimientos únicos (N = 12.189)")
b_tn = ax1.bar(x_pos + ancho/2, pct_ton.values, width=ancho, color=[GRIS, GRIS, GRANATE], label="% del volumen nacional (1.942.051,2 t)")

for idx, (bar_p, bar_t) in enumerate(zip(b_pl, b_tn)):
    vp = pct_plantas.values[idx]
    vt = pct_ton.values[idx]
    np_g = int(plantas_por_g.values[idx])
    dx_t = 0.05 if idx < 2 else 0.0
    ax1.text(bar_p.get_x() + bar_p.get_width()/2, vp + 1.5, f"{fmt_pct(vp)}\n({fmt_miles(np_g)})", ha="center", va="bottom", fontsize=8.3, color=AZUL, fontweight="bold")
    ax1.text(bar_t.get_x() + bar_t.get_width()/2 + dx_t, vt + 1.5, f"{fmt_pct(vt)}\n({fmt_dec(ton_por_g.values[idx]/1000, 1)} mil t)", ha="center", va="bottom", fontsize=8.2, color=GRANATE if idx==2 else "#333333", fontweight="bold")

ax1.set_xticks(x_pos)
ax1.set_xticklabels(etiquetas_tray, fontsize=9.2)
ax1.set_ylim(0, 114)
ax1.set_ylabel("Participación porcentual (%)")
ax1.set_title("A. Peso en el padrón de plantas vs. peso en el tonelaje total", loc="left", pad=10)
ax1.grid(axis="y", alpha=0.6)
ax1.legend(loc="upper left", frameon=True, facecolor="white", fontsize=8.4)

# Panel B: Curva de Lorenz de concentración por establecimiento (Núcleo persistente vs Padrón total)
ax2.plot([0, 100], [0, 100], linestyle="--", color=GRIS, linewidth=1.1, label="Equidistribución (Gini = 0,0)")
ax2.plot(
    conc_total["x_lorenz"] * 100, conc_total["y_lorenz"] * 100,
    color=AZUL, linewidth=2.0, linestyle="-.",
    label=f"Padrón total (N = {fmt_miles(conc_total['n_establecimientos'])}, Gini = {fmt_dec(conc_total['gini'], 4)})"
)
ax2.plot(
    conc_nucleo["x_lorenz"] * 100, conc_nucleo["y_lorenz"] * 100,
    color=GRANATE, linewidth=2.6,
    label=f"Núcleo persistente (n = {fmt_miles(conc_nucleo['n_establecimientos'])}, Gini = {fmt_dec(conc_nucleo['gini'], 4)})"
)
ax2.fill_between(conc_nucleo["x_lorenz"] * 100, conc_nucleo["y_lorenz"] * 100, conc_nucleo["x_lorenz"] * 100, color=GRANATE, alpha=0.10)

# Puntos destacados para el 10% y 1% superior dentro del núcleo persistente
x_90, y_90 = 90.0, 100.0 - conc_nucleo["share_top10_pct"]
x_99, y_99 = 99.0, 100.0 - conc_nucleo["share_top1_pct"]
ax2.scatter([x_90, x_99], [y_90, y_99], color=GRANATE, s=55, zorder=5)

caja_conc = (
    f"Concentración dentro del núcleo persistente (6.785 plantas = 1.858,0 mil t):\n"
    f"• Top 1 % ({conc_nucleo['n_top1_pct']} plantas): {fmt_pct(conc_nucleo['share_top1_pct'])} del núcleo ({fmt_dec(conc_nucleo['ton_top1_pct']/1000, 1)} mil t)\n"
    f"• Top 5 % ({conc_nucleo['n_top5_pct']} plantas): {fmt_pct(conc_nucleo['share_top5_pct'])} del núcleo ({fmt_dec(conc_nucleo['ton_top5_pct']/1000, 1)} mil t)\n"
    f"• Top 10 % ({conc_nucleo['n_top10_pct']} plantas): {fmt_pct(conc_nucleo['share_top10_pct'])} del núcleo ({fmt_dec(conc_nucleo['ton_top10_pct']/1000, 1)} mil t)\n"
    f"• Mediana trienal por planta del núcleo: {fmt_dec(conc_nucleo['mediana'], 1)} t (RIC: {fmt_dec(conc_nucleo['q25'], 1)} a {fmt_dec(conc_nucleo['q75'], 1)} t)\n"
    f"• En el padrón total (12.189 plantas): Top 1 % (122 plantas) = {fmt_pct(conc_total['share_top1_pct'])}; Top 10 % = {fmt_pct(conc_total['share_top10_pct'])}"
)
ax2.text(
    3.5, 46.0, caja_conc, fontsize=8.3, color="#0A1526",
    bbox={"boxstyle": "round,pad=0.45", "facecolor": HIELO, "edgecolor": GRANATE, "linewidth": 1.0}
)
ax2.annotate(
    f"90 % inferior acumula {fmt_pct(y_90)}\n(Top 10 % = {fmt_pct(conc_nucleo['share_top10_pct'])})",
    xy=(x_90, y_90), xytext=(52, 16),
    arrowprops={"arrowstyle": "->", "color": GRANATE, "lw": 1.2},
    fontsize=8.2, fontweight="bold", color=GRANATE
)
ax2.set_xlim(0, 102)
ax2.set_ylim(0, 102)
ax2.set_xlabel("Porcentaje acumulado de establecimientos (de menor a mayor volumen, %)")
ax2.set_ylabel("Porcentaje acumulado del volumen declarado (%)")
ax2.set_title("B. Curva de Lorenz y concentración en el 1 % y 10 % superior", loc="left", pad=10)
ax2.grid(True, alpha=0.6)
ax2.legend(loc="upper left", frameon=True, facecolor="white", fontsize=8.3)

fig1.text(
    0.01, -0.02,
    "Fuente: Elaboración propia a partir de artefactos de Fase 3 (RETC–SIDREP 2023–2025, N = 12.189 establecimientos únicos id_vu y 118.141 declaraciones).",
    fontsize=8.3, color="#4A5568"
)
plt.tight_layout(rect=[0, 0.01, 1, 0.94])
plt.show()

# --- FIGURA 2: Evolución y descomposición interanual (Panel A) y Concentración por rubro + desglose de Otras actividades (Panel B) ---
piv_anual = legible.pivot_table(index="año", columns="trayectoria_vu", values="cantidad_ton", aggfunc="sum")[orden_tray] / 1000.0
anios = [2023, 2024, 2025]

fig2, (ax2a, ax2b) = plt.subplots(1, 2, figsize=(13.8, 5.5), gridspec_kw={"width_ratios": [1.05, 1.15]})
fig2.suptitle(
    "El núcleo persistente (+30,9 mil t) compensa la caída no persistente (-2,1 mil t); Minería aporta el 41,3 % en solo 4.102 declaraciones",
    fontsize=12.2, fontweight="bold", y=0.99, color="#06304A"
)

x_a = np.arange(len(anios))
v_oca = piv_anual["Ocasional: entra o sale (1 año)"].values
v_int = piv_anual["Intermitente (2 años)"].values
v_per = piv_anual["Núcleo persistente (3 años)"].values
v_tot = v_oca + v_int + v_per

ax2a.bar(x_a, v_per, width=0.52, color=GRANATE, label="Núcleo persistente (3 años: +30,9 mil t)")
ax2a.bar(x_a, v_int, width=0.52, bottom=v_per, color=AZUL, label="Intermitente (2 años: -12,0 mil t)")
ax2a.bar(x_a, v_oca, width=0.52, bottom=v_per + v_int, color=GRIS, label="Ocasional (1 año: +9,9 mil t)")

for idx in range(len(anios)):
    ax2a.text(x_a[idx], v_tot[idx] + 14, f"Total: {fmt_dec(v_tot[idx], 1)} mil t", ha="center", va="bottom", fontsize=8.8, fontweight="bold", color="#06304A")
    ax2a.text(x_a[idx], v_per[idx]/2, f"Núcleo:\n{fmt_dec(v_per[idx], 1)} mil t\n({fmt_pct(v_per[idx]/v_tot[idx]*100)})", ha="center", va="center", fontsize=8.4, fontweight="bold", color="white")

caja_desc = (
    "Descomposición exacta del cambio (2023 → 2025: +28,7 mil t = +4,5 %):\n"
    "• Por trayectoria: Núcleo (+30,9 mil t) compensa baja no persistente (-2,1 mil t).\n"
    "• Par 2023→2024 (-5,0 mil t): Persist. -4,9 + Entrantes +9,0 - Salientes 9,1 mil t.\n"
    "• Par 2024→2025 (+33,7 mil t): Persist. +27,1 + Entrantes +20,0 - Salientes 13,4 mil t.\n"
    "• Sensibilidad: Sin Top 1 % (122 plantas) el resto del país crece +10,2 mil t."
)
ax2a.text(
    -0.42, 1045, caja_desc, fontsize=7.8, color="#0A1526", va="top",
    bbox={"boxstyle": "round,pad=0.36", "facecolor": HIELO, "edgecolor": AZUL, "linewidth": 0.9}
)
ax2a.set_xticks(x_a)
ax2a.set_xticklabels([str(a) for a in anios], fontsize=9.5, fontweight="bold")
ax2a.set_ylim(0, 1085)
ax2a.set_ylabel("Volumen anual declarado (miles de toneladas, desde cero)")
ax2a.set_title("A. Evolución anual y descomposición del cambio neto (2023–2025)", loc="left", pad=10)
ax2a.grid(axis="y", alpha=0.6)
ax2a.legend(loc="lower right", frameon=True, facecolor="white", fontsize=8.0)

# Panel B: Top 8 rubros por volumen + proporción de declaraciones y desagregación de "Otras actividades"
res_rubro = legible.groupby("rubro_nombre", as_index=False).agg(
    ton=("cantidad_ton", "sum"),
    n_decl=("cantidad_ton", "count"),
    n_plantas=("id_vu", "nunique"),
).sort_values("ton", ascending=False)
top8_rub = res_rubro.head(8).iloc[::-1].copy()
top8_rub["mil_ton"] = top8_rub["ton"] / 1000.0
top8_rub["pct_ton"] = top8_rub["ton"] / legible["cantidad_ton"].sum() * 100.0
top8_rub["pct_decl"] = top8_rub["n_decl"] / len(legible) * 100.0

colores_rub = [
    GRANATE if r == "Minería" else (AZUL if r in ("Otras actividades", "Gestores de residuos") else GRIS)
    for r in top8_rub["rubro_nombre"]
]
y_r = np.arange(len(top8_rub))
bars_r = ax2b.barh(y_r, top8_rub["mil_ton"], color=colores_rub, height=0.64)

for bar, (_, r) in zip(bars_r, top8_rub.iterrows()):
    txt_r = f"{fmt_dec(r['mil_ton'], 1)} mil t ({fmt_pct(r['pct_ton'])}) | {fmt_miles(r['n_decl'])} decl. ({fmt_pct(r['pct_decl'])})"
    ax2b.text(
        bar.get_width() + 12, bar.get_y() + bar.get_height()/2, txt_r,
        va="center", ha="left", fontsize=7.9,
        fontweight="bold" if r["rubro_nombre"] in ("Minería", "Otras actividades") else "normal",
        color=GRANATE if r["rubro_nombre"] == "Minería" else "#1A202C"
    )

caja_otras = (
    "¿Qué contiene «Otras actividades» (294,6 mil t | 41.589 decl.)?\n"
    "En la taxonomía RETC (id_rubro=20), CIIU4 revela 3 subsectores:\n"
    "1. Logística, almacenamiento y transporte ferroviario/vial: 151,1 mil t\n"
    "2. Hospitales y clínicas públicas/privadas: 35,3 mil t (12.793 decl.)\n"
    "3. Descontaminación y redes de gas/energía: 21,8 mil t\n"
    "En contraste, Minería aporta 41,3 % de la masa en solo 3,5 % de decl."
)
ax2b.text(
    755, 0.10, caja_otras, fontsize=7.6, color="#0A1526", va="bottom",
    bbox={"boxstyle": "round,pad=0.36", "facecolor": HIELO, "edgecolor": AZUL, "linewidth": 0.9}
)

ax2b.set_yticks(y_r)
ax2b.set_yticklabels(top8_rub["rubro_nombre"], fontsize=8.6)
ax2b.set_xlim(0, 1680)
ax2b.set_xlabel("Volumen acumulado 2023–2025 (miles de toneladas)")
ax2b.set_title("B. Top 8 rubros: desproporción de Minería y desglose de Otras actividades", loc="left", pad=10)
ax2b.grid(axis="x", alpha=0.6)

fig2.text(
    0.01, -0.02,
    "Fuente: Elaboración propia a partir de artefactos de Fase 3 y desglose CIIU4 de id_rubro=20 (RETC–SIDREP 2023–2025, N = 118.141 declaraciones).",
    fontsize=8.3, color="#4A5568"
)
plt.tight_layout(rect=[0, 0.01, 1, 0.94])
plt.show()


## 5. Figuras 3 y 4: Gradiente Regional por Establecimiento Independiente (`N = 12.189`) y Doble Control de Confusión (`OE3`)

### Justificación de la unidad de análisis (`N = 12.189` establecimientos únicos `id_vu`)
En un registro administrativo donde un mismo establecimiento puede enviar docenas de declaraciones al año (una por corriente de residuo `id_lista_a`), calcular proporciones e intervalos de Wilson sobre las `118.141` declaraciones viola el supuesto de independencia estadística, estrecha artificialmente los intervalos de confianza y sobrepondera a las plantas con más trámites. Por ello, en las **Figuras 3 y 4** adoptamos como **unidad de análisis primaria al establecimiento único independiente (`id_vu`, `N = 12.189`)**, clasificando como **Gran Generador (`>= 12 t/año`, Art. 25 D.S. N° 148)** a toda planta cuya generación anual consolidada alcanzó o superó las 12 toneladas en al menos un año del trienio (`2.951` de `12.189` plantas = **`24,2 %` promedio nacional**; a modo de referencia complementaria, a nivel de `establecimiento-año` [`N = 28.099`] la tasa nacional es **`22,8 %`**).

### Lectura en 4 enunciados de la Figura 3 (`OE3` — Resolución territorial con criterio estadístico y efecto sin Minería)
1. **Qué muestra:** El Panel A presenta para las 16 regiones la proporción de establecimientos únicos (`N = 12.189`) que son Grandes Generadores (`>= 12 t/año`) con intervalos de confianza de Wilson al 95 %, asignando el color **granate** con un **criterio estrictamente estadístico** (regiones cuyo IC al 95 % queda completamente sobre el promedio nacional de `24,2 %`) y superponiendo en rombos dorados la tasa regional **excluyendo al rubro Minería**; el Panel B muestra la proporción regional de establecimientos pertenecientes al **núcleo persistente (3 años)** con sus respectivos IC de Wilson al 95 % (promedio nacional `55,7 %`).
2. **Qué se infiere:** **Antofagasta (`48,0 %`, IC `[43,8 %; 52,3 %]`, `n = 535`)** y la **Región Metropolitana (`32,7 %`, IC `[31,2 %; 34,2 %]`, `n = 3.664`)** superan con su intervalo completo la media nacional (`24,2 %`), mientras que **Arica y Parinacota (`27,5 %`)**, **Atacama (`26,9 %`)** y **Tarapacá (`24,4 %`)** se sitúan sobre el promedio puntual con intervalos que lo incluyen (y a nivel de `establecimiento-año`, `N = 28.099`, tanto **Arica [`27,8 %`]** como **Atacama [`27,5 %`]** quedan también con su IC 95 % íntegramente sobre el promedio nacional de `22,8 %`). Al retirar la Minería (rombos dorados), la tasa cae **`5,7 pp` en Atacama (`26,9 % → 21,2 %`)**, **`3,3 pp` en Antofagasta (`48,0 % → 44,7 %`)**, **`2,7 pp` en Coquimbo (`18,4 % → 15,7 %`)** y **`2,4 pp` en Tarapacá (`24,4 % → 22,0 %`)**, demostrando que en Antofagasta incluso la industria no minera (logística, química y servicios de soporte) opera a gran escala y con la mayor persistencia trienal del país (**`71,2 %`, IC `[67,2 %; 74,9 %]`** en el Panel B).
3. **Qué límite tiene:** La ubicación regional corresponde a la dirección de la instalación registrada en Ventanilla Única, no al lugar de disposición final del residuo.
4. **Cómo aporta al relato:** Reemplaza umbrales arbitrarios por inferencia de intervalos de confianza sobre unidades independientes y separa cuánto del gradiente regional proviene directamente de faenas mineras frente al tejido industrial regional.

### Lectura en 4 enunciados de la Figura 4 (`OE3` — Doble control de confusión: Minería y Macrozona Norte)
1. **Qué muestra:** El Panel A evalúa la tasa de Gran Generador por establecimiento (`N = 12.189`) según persistencia (`No persistente 1–2 años` vs. `Núcleo persistente 3 años`) controlando por **Rubro (`Minería` vs. `Otros 18 rubros no mineros`)**; el Panel B incorpora la **Región (`Macrozona Norte Minera` [Arica, Tarapacá, Antofagasta, Atacama y Coquimbo] vs. `Resto del país` [11 regiones]) como segundo control simultáneo**.
2. **Qué se infiere:** La persistencia trienal **se asocia** de manera robusta y estadísticamente significativa con una mayor proporción de Gran Generador en todos los estratos: en el Panel A, dentro de **Minería (`n = 271` plantas)** sube de **`19,8 %` (`IC [12,7 %; 29,4 %]`)** a **`73,5 %` (`IC [66,7 %; 79,3 %]`, `3,7×`)**, y en los **Otros 18 rubros (`n = 11.918` plantas)** sube de **`9,0 %` (`IC [8,3 %; 9,8 %]`)** a **`35,1 %` (`IC [34,0 %; 36,3 %]`, `3,9×`)**. En el Panel B, al controlar simultáneamente por **Macrozona Norte y Rubro**, la asociación se mantiene intacta tanto en los **rubros no mineros del Norte (`40,4 % vs 11,7 %`, `3,5×`)** como en los **rubros no mineros del Resto del país (`34,3 % vs 8,6 %`, `4,0×`, `n = 10.177` plantas)** y en la **Minería del Norte (`85,2 % vs 30,6 %`) y del Resto del país (`57,1 % vs 5,4 %`)**.
3. **Qué límite tiene:** La persistencia administrativa no es la *causa* de la gran generación; lo más probable es que ambas variables reflejen una causa común latente —la **escala estructural de la operación industrial**—, por lo que documentamos una asociación estratificada y no un efecto causal.
4. **Cómo aporta al relato:** Descarta mediante doble estratificación (`Persistencia × Minería × Macrozona`) que el vínculo entre permanencia en el padrón y gran generación sea un artefacto de confusión sectorial o territorial.


In [ ]:
# Celda In[6]: Generación de Figura 3 (Gradiente regional por establecimiento con criterio IC 95% y sin Minería) y Figura 4 (Doble control)
# --- FIGURA 3: Panel A (Tasa regional de Gran Generador por establecimiento N=12.189 con y sin Minería) y Panel B (Persistencia regional) ---
p_nac_gg = float(df_est["es_gran_generador_planta"].mean())
t_reg_gg = tasa_por_grupo(df_est, "region", "es_gran_generador_planta").sort_values("tasa", ascending=True).reset_index(drop=True)
t_reg_nomin = tasa_por_grupo(df_est[df_est["rubro_nombre"] != "Minería"], "region", "es_gran_generador_planta").set_index("region")["tasa"]
t_reg_gg["tasa_sin_mineria"] = t_reg_gg["region"].map(t_reg_nomin)

p_nac_pers = float(df_est["es_nucleo_persistente"].mean())
t_reg_pers = tasa_por_grupo(df_est, "region", "es_nucleo_persistente").set_index("region").loc[t_reg_gg["region"]].reset_index()

abreviaturas_reg = {
    "Metropolitana de Santiago": "Metropolitana de Santiago",
    "Libertador General Bernardo O'Higgins": "L. G. Bernardo O'Higgins",
    "Magallanes y de la Antártica Chilena": "Magallanes y Antártica",
    "Aysén del General Carlos Ibáñez del Campo": "Aysén del G. C. Ibáñez",
}
etiquetas_reg = [abreviaturas_reg.get(r, r) for r in t_reg_gg["region"]]

fig3, (ax3a, ax3b) = plt.subplots(1, 2, figsize=(14.2, 6.2), sharey=True, gridspec_kw={"width_ratios": [1.22, 0.98]})
fig3.suptitle(
    "Por establecimiento (N = 12.189), Antofagasta (48,0 %) y RM (32,7 %) superan con su IC 95 % el promedio nacional de Gran Generador (24,2 %)",
    fontsize=12.0, fontweight="bold", y=0.99, color="#06304A"
)

y_pos = np.arange(len(t_reg_gg))
colores_f3a = [
    GRANATE if r["ic_inferior"] > p_nac_gg else (GRIS if r["ic_superior"] < p_nac_gg else AZUL)
    for _, r in t_reg_gg.iterrows()
]

ax3a.barh(y_pos, t_reg_gg["tasa"] * 100, color=colores_f3a, height=0.64, alpha=0.92)
err_inf_a = (t_reg_gg["tasa"] - t_reg_gg["ic_inferior"]) * 100
err_sup_a = (t_reg_gg["ic_superior"] - t_reg_gg["tasa"]) * 100
ax3a.errorbar(
    t_reg_gg["tasa"] * 100, y_pos, xerr=[err_inf_a, err_sup_a],
    fmt="none", ecolor="#1A202C", elinewidth=1.25, capsize=2.8, zorder=4
)
# Marcadores de la tasa regional excluyendo Minería
ax3a.scatter(
    t_reg_gg["tasa_sin_mineria"] * 100, y_pos,
    marker="D", s=38, facecolor=DORADO, edgecolor="#1A202C", linewidth=0.8, zorder=6,
    label="Tasa regional excluyendo Minería"
)
ax3a.axvline(p_nac_gg * 100, color=GRANATE, linestyle="--", linewidth=1.2, label=f"Promedio nacional: {fmt_pct(p_nac_gg*100)}")

# Etiquetas ubicadas después del límite superior del IC (cero superposición con barras de error)
for idx, r in t_reg_gg.iterrows():
    x_txt = r["ic_superior"] * 100 + 0.9
    dif_min = (r["tasa"] - r["tasa_sin_mineria"]) * 100
    nota_min = f" (sin Min.: {fmt_pct(r['tasa_sin_mineria']*100)})" if abs(dif_min) >= 2.0 else ""
    ax3a.text(
        x_txt, idx,
        f"{fmt_pct(r['tasa']*100)} (n={fmt_miles(r['n'])}){nota_min}",
        va="center", ha="left", fontsize=7.8,
        fontweight="bold" if r["ic_inferior"] > p_nac_gg else "normal",
        color=GRANATE if r["ic_inferior"] > p_nac_gg else "#1A202C"
    )

from matplotlib.patches import Patch
from matplotlib.lines import Line2D
leyenda_f3a = [
    Patch(facecolor=GRANATE, label="IC 95 % completamente sobre promedio nacional (> 24,2 %)"),
    Patch(facecolor=AZUL, label="IC 95 % incluye al promedio nacional (24,2 %)"),
    Patch(facecolor=GRIS, label="IC 95 % completamente bajo promedio nacional (< 24,2 %)"),
    Line2D([0], [0], marker="D", color="w", markerfacecolor=DORADO, markeredgecolor="#1A202C", markersize=6, label="Tasa regional sin Minería (efecto composicional)"),
]
ax3a.set_yticks(y_pos)
ax3a.set_yticklabels(etiquetas_reg, fontsize=8.8)
ax3a.set_xlim(0, 74)
ax3a.set_xlabel("Establecimientos Gran Generador (>= 12 t/año en al menos 1 año, %)")
ax3a.set_title("A. Tasa regional de Gran Generador con y sin Minería (IC Wilson 95 %)", loc="left", pad=10)
ax3a.grid(axis="x", alpha=0.6)
ax3a.legend(handles=leyenda_f3a, loc="lower right", frameon=True, facecolor="white", fontsize=7.7)

# Panel B: Proporción regional de establecimientos del núcleo persistente (3 años)
colores_f3b = [
    GRANATE if r["ic_inferior"] > p_nac_pers else (GRIS if r["ic_superior"] < p_nac_pers else AZUL)
    for _, r in t_reg_pers.iterrows()
]
ax3b.barh(y_pos, t_reg_pers["tasa"] * 100, color=colores_f3b, height=0.64, alpha=0.90)
err_inf_b = (t_reg_pers["tasa"] - t_reg_pers["ic_inferior"]) * 100
err_sup_b = (t_reg_pers["ic_superior"] - t_reg_pers["tasa"]) * 100
ax3b.errorbar(
    t_reg_pers["tasa"] * 100, y_pos, xerr=[err_inf_b, err_sup_b],
    fmt="none", ecolor="#1A202C", elinewidth=1.25, capsize=2.8, zorder=4
)
ax3b.axvline(p_nac_pers * 100, color=GRANATE, linestyle="--", linewidth=1.2, label=f"Persistencia nacional: {fmt_pct(p_nac_pers*100)}")

for idx, r in t_reg_pers.iterrows():
    x_txt_b = r["ic_superior"] * 100 + 0.9
    ax3b.text(
        x_txt_b, idx,
        f"{fmt_pct(r['tasa']*100)} [{fmt_pct(r['ic_inferior']*100)}; {fmt_pct(r['ic_superior']*100)}]",
        va="center", ha="left", fontsize=7.8,
        fontweight="bold" if r["ic_inferior"] > p_nac_pers else "normal",
        color=GRANATE if r["ic_inferior"] > p_nac_pers else "#1A202C"
    )

ax3b.set_xlim(0, 102)
ax3b.set_ylim(-1.25, 15.8)
ax3b.set_xlabel("Establecimientos en el núcleo persistente (declaran los 3 años, %)")
ax3b.set_title("B. Proporción regional de persistencia trienal (IC Wilson 95 %)", loc="left", pad=10)
ax3b.grid(axis="x", alpha=0.6)
ax3b.legend(loc="lower right", frameon=True, facecolor="white", fontsize=8.0)

fig3.text(
    0.01, -0.02,
    "Fuente: Elaboración propia sobre N = 12.189 establecimientos únicos independientes (id_vu, RETC–SIDREP 2023–2025). En establecimiento-año (N = 28.099), Arica (27,8 %) y Atacama (27,5 %) también superan con su IC 95 % el promedio nacional (22,8 %).",
    fontsize=8.1, color="#4A5568"
)
plt.tight_layout(rect=[0, 0.01, 1, 0.94])
plt.show()

# --- FIGURA 4: Doble control de confusión: Persistencia x Minería (Panel A) y Persistencia x Minería x Macrozona Norte vs Resto (Panel B) ---
fig4, (ax4a, ax4b) = plt.subplots(1, 2, figsize=(14.0, 5.5), gridspec_kw={"width_ratios": [0.95, 1.25]})
fig4.suptitle(
    "La persistencia trienal se asocia a mayor tasa de Gran Generador al controlar por Minería (3,7×) y Macrozona Norte (3,5× a 4,0×)",
    fontsize=12.0, fontweight="bold", y=0.99, color="#06304A"
)

# Panel A: Primer control por Rubro (Minería vs Otros 18 rubros no mineros) sobre N = 12.189 establecimientos
sectores = ["Otros 18 rubros no mineros", "Minería"]
grupos_p = ["No persistente (1–2 años)", "Núcleo persistente (3 años)"]
x_s = np.arange(len(sectores))
w_s = 0.34

for idx_g, (g_nom, col_g, desp) in enumerate([(grupos_p[0], AZUL, -w_s/2), (grupos_p[1], GRANATE, w_s/2)]):
    tasas_g, inf_g, sup_g, ns_g = [], [], [], []
    for sec in sectores:
        sub = df_est[(df_est["sector_minero_bin"] == sec) & (df_est["grupo_persistencia_bin"] == g_nom)]
        n_s = len(sub)
        c_s = int(sub["es_gran_generador_planta"].sum())
        ic_l, ic_u = intervalo_wilson(c_s, n_s)
        t_val = (c_s / n_s) * 100.0
        tasas_g.append(t_val)
        inf_g.append(t_val - ic_l * 100.0)
        sup_g.append(ic_u * 100.0 - t_val)
        ns_g.append(n_s)
    bars_a = ax4a.bar(x_s + desp, tasas_g, width=w_s, color=col_g, label=g_nom, alpha=0.93)
    ax4a.errorbar(x_s + desp, tasas_g, yerr=[inf_g, sup_g], fmt="none", ecolor="#1A202C", elinewidth=1.3, capsize=3.5, zorder=5)
    for b_i, t_v, e_u, n_v in zip(bars_a, tasas_g, sup_g, ns_g):
        ax4a.text(
            b_i.get_x() + b_i.get_width()/2, t_v + e_u + 1.8,
            f"{fmt_pct(t_v)}\n(n={fmt_miles(n_v)})",
            ha="center", va="bottom", fontsize=8.3, fontweight="bold", color=col_g
        )

ax4a.annotate("Razón = 3,9×\n(IC no solapados)", xy=(0.17, 35.1), xytext=(0.0, 58), arrowprops={"arrowstyle": "->", "color": GRANATE, "lw": 1.2}, ha="center", fontsize=8.4, fontweight="bold", color=GRANATE)
ax4a.annotate("Razón = 3,7×\n(IC no solapados)", xy=(1.17, 73.5), xytext=(0.82, 83), arrowprops={"arrowstyle": "->", "color": GRANATE, "lw": 1.2}, ha="center", fontsize=8.4, fontweight="bold", color=GRANATE)

ax4a.set_xticks(x_s)
ax4a.set_xticklabels(["Otros 18 rubros no mineros\n(N = 11.918 plantas)", "Rubro Minería\n(N = 271 plantas)"], fontsize=9.2)
ax4a.set_ylim(0, 98)
ax4a.set_ylabel("Establecimientos Gran Generador (>= 12 t/año, %)")
ax4a.set_title("A. Control 1: Persistencia × Rubro (N = 12.189 plantas)", loc="left", pad=10)
ax4a.grid(axis="y", alpha=0.6)
ax4a.legend(loc="upper left", frameon=True, facecolor="white", fontsize=8.3)

# Panel B: Segundo control simultáneo (Macrozona Norte Minera vs Resto del país x Rubro x Persistencia)
estratos_b = [
    ("Resto del país (11 reg.)", "Otros 18 rubros no mineros", "Resto del país\nOtros 18 rubros\n(N = 10.177)"),
    ("Macrozona Norte Minera (5 reg.)", "Otros 18 rubros no mineros", "Macrozona Norte\nOtros 18 rubros\n(N = 1.741)"),
    ("Resto del país (11 reg.)", "Minería", "Resto del país\nMinería\n(N = 114)"),
    ("Macrozona Norte Minera (5 reg.)", "Minería", "Macrozona Norte\nMinería\n(N = 157)"),
]
x_b = np.arange(len(estratos_b))
w_b = 0.34

for idx_g, (g_nom, col_g, desp) in enumerate([(grupos_p[0], AZUL, -w_b/2), (grupos_p[1], GRANATE, w_b/2)]):
    tasas_b, inf_b, sup_b, ns_b = [], [], [], []
    for mz, sec, _ in estratos_b:
        sub = df_est[(df_est["macrozona"] == mz) & (df_est["sector_minero_bin"] == sec) & (df_est["grupo_persistencia_bin"] == g_nom)]
        n_s = len(sub)
        c_s = int(sub["es_gran_generador_planta"].sum())
        ic_l, ic_u = intervalo_wilson(c_s, n_s)
        t_val = (c_s / n_s) * 100.0
        tasas_b.append(t_val)
        inf_b.append(t_val - ic_l * 100.0)
        sup_b.append(ic_u * 100.0 - t_val)
        ns_b.append(n_s)
    bars_b = ax4b.bar(x_b + desp, tasas_b, width=w_b, color=col_g, label=g_nom, alpha=0.93)
    ax4b.errorbar(x_b + desp, tasas_b, yerr=[inf_b, sup_b], fmt="none", ecolor="#1A202C", elinewidth=1.3, capsize=3.2, zorder=5)
    for b_i, t_v, e_u, n_v in zip(bars_b, tasas_b, sup_b, ns_b):
        ax4b.text(
            b_i.get_x() + b_i.get_width()/2, t_v + e_u + 1.6,
            f"{fmt_pct(t_v)}\n(n={fmt_miles(n_v)})",
            ha="center", va="bottom", fontsize=7.8, fontweight="bold", color=col_g
        )

caja_doble = (
    "Hallazgo del doble control (Región × Rubro × Persistencia):\n"
    "• Fuera de la Minería, el núcleo persistente cuadruplica la tasa de Gran Generador\n"
    "  tanto en el Resto del país (34,3 % vs 8,6 % = 4,0×) como en el Norte (40,4 % vs 11,7 % = 3,5×).\n"
    "• Explicación estructural: la persistencia no causa gran generación; ambas se asocian\n"
    "  porque reflejan la escala permanente de operación de las plantas industriales mayores."
)
ax4b.text(
    -0.42, 114, caja_doble, fontsize=7.8, color="#0A1526", va="top",
    bbox={"boxstyle": "round,pad=0.36", "facecolor": HIELO, "edgecolor": GRANATE, "linewidth": 0.9}
)

ax4b.set_xticks(x_b)
ax4b.set_xticklabels([e[2] for e in estratos_b], fontsize=8.6)
ax4b.set_ylim(0, 118)
ax4b.set_ylabel("Establecimientos Gran Generador (>= 12 t/año, %)")
ax4b.set_title("B. Control 2: Persistencia × Rubro × Macrozona Norte vs. Resto del país", loc="left", pad=10)
ax4b.grid(axis="y", alpha=0.6)
ax4b.legend(loc="lower right", frameon=True, facecolor="white", fontsize=8.0)

fig4.text(
    0.01, -0.02,
    "Fuente: Elaboración propia sobre N = 12.189 establecimientos únicos (id_vu, RETC–SIDREP 2023–2025). Macrozona Norte Minera comprende Arica y Parinacota, Tarapacá, Antofagasta, Atacama y Coquimbo.",
    fontsize=8.2, color="#4A5568"
)
plt.tight_layout(rect=[0, 0.01, 1, 0.94])
plt.show()

# Verificación analítica del Factor de Mentira de Tufte (documentado en código sin ocupar un panel de figura)
v_2023, v_2025 = float(v_tot[0]), float(v_tot[2])
cambio_real_datos = (v_2025 - v_2023) / v_2023
base_truncada = 630.0
cambio_visual_truncado = ((v_2025 - base_truncada) - (v_2023 - base_truncada)) / (v_2023 - base_truncada)
factor_mentira_truncado = cambio_visual_truncado / cambio_real_datos
print(
    f"[Verificación de Tufte en notebook] Variación real 2023→2025: +{fmt_pct(cambio_real_datos*100, 2)} | "
    f"Variación aparente truncando eje Y en 630 mil t: +{fmt_pct(cambio_visual_truncado*100, 1)} | "
    f"Factor de Mentira = {fmt_dec(factor_mentira_truncado, 1)} (frente a 1,0 con eje desde cero en Figura 2A)."
)


## 6. Validación en Tres Escenarios, Persistencia de Figuras a `200 dpi` y Tabla de Hallazgos con Intervalos de Confianza y Celda de Origen
Cerramos el cuaderno verificando los tres escenarios de prueba (`normal`, `límite` y `excepción`), ejecutando la suite completa de **`21` pruebas unitarias en `pytest`** (`14` de Fase 3 + `7` de Fase 4, incluyendo descomposición interanual exacta y curva de Lorenz), guardando las cuatro figuras a `200 dpi` en `F4/figuras/` y exportando `F4/docs/hallazgos_f4.csv` con las columnas **`ic_wilson_95`** y **`celda_notebook`** para asegurar consistencia total entre el notebook, el informe y el video.


In [ ]:
# Celda In[7]: Controles en 3 escenarios, ejecución de pytest (21 passed), guardado a 200 dpi y tabla de hallazgos con IC 95% y celda
import subprocess

# 1. Verificación explícita en los 3 escenarios (Normal, Límite y Excepción)
ic_lim_0 = intervalo_wilson(0, 50)
ic_lim_1 = intervalo_wilson(50, 50)
assert 0.0 <= ic_lim_0[0] < ic_lim_0[1] <= 1.0 and 0.0 <= ic_lim_1[0] < ic_lim_1[1] <= 1.0

excepcion_capturada = False
try:
    cargar_artefactos(RAIZ_PROYECTO / "carpeta_inexistente_f3")
except FileNotFoundError:
    excepcion_capturada = True
assert excepcion_capturada, "cargar_artefactos no lanzó FileNotFoundError ante carpeta inexistente."

tabla_validacion_f4 = pd.DataFrame([
    {"Escenario": "1. Normal (Datos reales F3→F4)", "Prueba Ejecutada": "Reversión np.allclose (118.141 filas), SHA-256 e identidad de descomposición 2023–2025", "Resultado": f"OK (dif. {dif_max_kg:.1e} kg)"},
    {"Escenario": "2. Valores límite (0/50 y 50/50)", "Prueba Ejecutada": "intervalo_wilson(0, 50) e intervalo_wilson(50, 50) dentro de [0, 1] sin NaN", "Resultado": f"OK ({ic_lim_0} y {ic_lim_1})"},
    {"Escenario": "3. Excepción controlada", "Prueba Ejecutada": "cargar_artefactos() sobre ruta sin CSV y One-Hot con suma != 1", "Resultado": "OK (FileNotFoundError / ValueError)"},
])
display(tabla_validacion_f4)

# 2. Ejecución de la suite pytest completa (21 passed)
res_pytest = subprocess.run([sys.executable, "-m", "pytest", str(RAIZ_PROYECTO / "tests"), "-q"], capture_output=True, text=True)
print("[OK] Salida de pytest:", res_pytest.stdout.strip())
assert res_pytest.returncode == 0, f"Fallo en pytest: {res_pytest.stdout}\n{res_pytest.stderr}"

# 3. Exportación de las 4 figuras a 200 dpi en F4/figuras/
CARPETA_FIGURAS = RAIZ_PROYECTO / "F4" / "figuras"
CARPETA_DOCS_F4 = RAIZ_PROYECTO / "F4" / "docs"
CARPETA_DOCS_F4.mkdir(parents=True, exist_ok=True)

r_f1 = guardar_figura(fig1, CARPETA_FIGURAS, "fig1_persistencia_vs_rotacion", dpi=200)
r_f2 = guardar_figura(fig2, CARPETA_FIGURAS, "fig2_evolucion_y_concentracion_rubros", dpi=200)
r_f3 = guardar_figura(fig3, CARPETA_FIGURAS, "fig3_gradiente_regional_gran_generador", dpi=200)
r_f4 = guardar_figura(fig4, CARPETA_FIGURAS, "fig4_control_confusion_y_factor_mentira", dpi=200)

for r_img in [r_f1, r_f2, r_f3, r_f4]:
    assert r_img.exists() and r_img.stat().st_size > 20_000, f"Figura no generada correctamente: {r_img}"

# 4. Construcción de F4/docs/hallazgos_f4.csv con intervalo de confianza (ic_wilson_95) y celda del notebook (celda_notebook)
ic_pers_nac = intervalo_wilson(6785, 12189)
ic_gg_nac_est = intervalo_wilson(int(df_est["es_gran_generador_planta"].sum()), len(df_est))
ic_gg_antof = intervalo_wilson(257, 535)
ic_gg_rm = intervalo_wilson(1198, 3664)
ic_gg_atac = intervalo_wilson(118, 438)
ic_gg_tarap = intervalo_wilson(64, 262)
ic_pers_antof = intervalo_wilson(381, 535)
ic_min_pers = intervalo_wilson(136, 185)
ic_min_nopers = intervalo_wilson(17, 86)
ic_nomin_pers = intervalo_wilson(2317, 6600)
ic_nomin_nopers = intervalo_wilson(481, 5318)
ic_norte_nomin_pers = intervalo_wilson(371, 919)
ic_norte_nomin_nopers = intervalo_wilson(96, 822)
ic_resto_nomin_pers = intervalo_wilson(1946, 5681)
ic_resto_nomin_nopers = intervalo_wilson(385, 4496)

hallazgos_df = pd.DataFrame([
    {
        "objetivo": "OE1",
        "figura": "Figura 1A",
        "celda_notebook": "In[4] / In[5]",
        "unidad_analisis": "Establecimiento único (id_vu, N = 12.189)",
        "indicador": "Proporción de establecimientos en el núcleo persistente (3 años: 2023–2025) y participación en el volumen nacional",
        "cifra_principal": "55,7 % de las plantas (6.785 de 12.189) concentran el 95,7 % del tonelaje nacional (1.857.961,4 t de 1.942.051,2 t)",
        "ic_wilson_95": f"[{fmt_pct(ic_pers_nac[0]*100)}; {fmt_pct(ic_pers_nac[1]*100)}] en proporción de plantas",
        "mediana_y_ric": "Mediana trienal del núcleo = 11,7 t (RIC: 2,6 a 47,6 t); por declaración = 267 kg (RIC: 60 a 1.656 kg)",
    },
    {
        "objetivo": "OE1",
        "figura": "Figura 1B",
        "celda_notebook": "In[4] / In[5]",
        "unidad_analisis": "Núcleo persistente (n = 6.785) y Padrón total (N = 12.189)",
        "indicador": "Concentración interna del volumen (Top 1 %, Top 10 % e índice de Gini)",
        "cifra_principal": "Dentro del núcleo persistente, el 1 % superior (68 plantas) concentra el 71,2 % (1.322,9 mil t) y el 10 % superior (678 plantas) el 92,0 % (1.708,8 mil t); Gini = 0,9421 (en el padrón total: Top 1 % = 75,4 %, Top 10 % = 94,1 %, Gini = 0,9567)",
        "ic_wilson_95": "Censo poblacional de masa (Gini = 0,9421 en núcleo; 0,9567 en padrón total)",
        "mediana_y_ric": "Mediana trienal total = 4,0 t (RIC: 0,8 a 21,0 t)",
    },
    {
        "objetivo": "OE2",
        "figura": "Figura 2A",
        "celda_notebook": "In[4] / In[5]",
        "unidad_analisis": "Flujo anual nacional y por par de años (2023→2024, 2024→2025 y 2023→2025)",
        "indicador": "Descomposición del cambio interanual (Persistentes + Entrantes - Salientes) y sensibilidad",
        "cifra_principal": "El volumen total creció +28.709,3 t (+4,5 %: de 639,4 mil a 668,1 mil t). El núcleo persistente creció +30.855,2 t, explicando todo el crecimiento neto y compensando la baja neta (-2.145,9 t) de intermitentes (-12.032,0 t) y ocasionales (+9.886,1 t). Sin el Top 1 % (122 plantas), el resto del país crece +10.246,6 t",
        "ic_wilson_95": "Identidad contable exacta verificada con assert (diferencia = 0,0 t)",
        "mediana_y_ric": "2023→2024 (-4.971,0 t = -4.870,6 + 8.972,2 - 9.072,6); 2024→2025 (+33.680,3 t = +27.076,6 + 19.989,9 - 13.386,2)",
    },
    {
        "objetivo": "OE2",
        "figura": "Figura 2B",
        "celda_notebook": "In[5]",
        "unidad_analisis": "Rubros productivos (19 rubros, N = 118.141 declaraciones)",
        "indicador": "Desproporción de Minería y desagregación CIIU4 de Otras actividades",
        "cifra_principal": "Minería aporta el 41,3 % del volumen nacional (801,1 mil t) en solo 4.102 declaraciones (3,5 % de los registros, 271 plantas). Otras actividades (294,6 mil t, 15,2 % en 41.589 decl.) agrupa logística/almacenamiento y transporte ferroviario/vial (151,1 mil t), hospitales públicos/privados (35,3 mil t en 12.793 decl.) y descontaminación/gas (21,8 mil t)",
        "ic_wilson_95": "Proporción de declaraciones mineras: 3,47 % [3,37 %; 3,58 %]",
        "mediana_y_ric": "Promedio por declaración en Minería = 195,3 t vs 7,1 t en Otras actividades",
    },
    {
        "objetivo": "OE3",
        "figura": "Figura 3A y 3B",
        "celda_notebook": "In[6]",
        "unidad_analisis": "Establecimiento único independiente (id_vu, N = 12.189)",
        "indicador": "Tasa regional de Gran Generador (>= 12 t/año) con y sin Minería, y persistencia regional",
        "cifra_principal": "Promedio nacional = 24,2 % (2.951/12.189 plantas). Superan con su IC 95 % completo la media nacional: Antofagasta (48,0 %; sin Minería 44,7 %) y RM (32,7 %; sin Minería 32,5 %), seguidas por Arica (27,5 %), Atacama (26,9 %; sin Minería 21,2 %) y Tarapacá (24,4 %; sin Minería 22,0 %). Antofagasta lidera en persistencia trienal con 71,2 % vs 55,7 % nacional",
        "ic_wilson_95": f"Nacional: [{fmt_pct(ic_gg_nac_est[0]*100)}; {fmt_pct(ic_gg_nac_est[1]*100)}] | Antofagasta: [{fmt_pct(ic_gg_antof[0]*100)}; {fmt_pct(ic_gg_antof[1]*100)}] | RM: [{fmt_pct(ic_gg_rm[0]*100)}; {fmt_pct(ic_gg_rm[1]*100)}] | Atacama: [{fmt_pct(ic_gg_atac[0]*100)}; {fmt_pct(ic_gg_atac[1]*100)}] | Persistencia Antof.: [{fmt_pct(ic_pers_antof[0]*100)}; {fmt_pct(ic_pers_antof[1]*100)}]",
        "mediana_y_ric": "En establecimiento-año (N = 28.099): promedio nacional = 22,8 %; Antofagasta (44,5 %), RM (30,9 %), Arica (27,8 %) y Atacama (27,5 %) quedan con su IC 95 % sobre la media",
    },
    {
        "objetivo": "OE3",
        "figura": "Figura 4A y 4B",
        "celda_notebook": "In[6]",
        "unidad_analisis": "Establecimiento único independiente (id_vu, N = 12.189)",
        "indicador": "Doble control de confusión: Persistencia × Minería (Panel A) y Persistencia × Minería × Macrozona Norte vs. Resto (Panel B)",
        "cifra_principal": "Control 1 (Rubro): en Minería sube de 19,8 % (no persistente) a 73,5 % (núcleo persistente, 3,7×); en Otros 18 rubros sube de 9,0 % a 35,1 % (3,9×). Control 2 (Macrozona × Rubro): en Otros 18 rubros del Norte sube de 11,7 % a 40,4 % (3,5×) y en Otros 18 rubros del Resto del país sube de 8,6 % a 34,3 % (4,0×)",
        "ic_wilson_95": f"Minería núcleo: [{fmt_pct(ic_min_pers[0]*100)}; {fmt_pct(ic_min_pers[1]*100)}] vs no pers.: [{fmt_pct(ic_min_nopers[0]*100)}; {fmt_pct(ic_min_nopers[1]*100)}] | Otros 18 núcleo: [{fmt_pct(ic_nomin_pers[0]*100)}; {fmt_pct(ic_nomin_pers[1]*100)}] vs no pers.: [{fmt_pct(ic_nomin_nopers[0]*100)}; {fmt_pct(ic_nomin_nopers[1]*100)}] | Norte no min. núcleo: [{fmt_pct(ic_norte_nomin_pers[0]*100)}; {fmt_pct(ic_norte_nomin_pers[1]*100)}] | Resto no min. núcleo: [{fmt_pct(ic_resto_nomin_pers[0]*100)}; {fmt_pct(ic_resto_nomin_pers[1]*100)}]",
        "mediana_y_ric": "Factor de Mentira de Tufte verificado en In[6]: 67,8 al truncar eje Y en 630 mil t vs 1,0 desde cero",
    },
])
ruta_hallazgos = CARPETA_DOCS_F4 / "hallazgos_f4.csv"
hallazgos_df.to_csv(ruta_hallazgos, index=False, encoding="utf-8")
display(hallazgos_df)

metadatos_f4 = {
    "fase": "Fase 4 — Consolidación, visualización y comunicación de resultados",
    "integrantes_activos": ["Victoria Álvarez Pinto", "Ignacio Peña Pizarro", "Raúl Hormazábal Figueroa"],
    "python_version": info_entorno["python_version"],
    "sha256_base_maestra": sha256_verificado,
    "n_declaraciones": int(len(legible)),
    "n_establecimientos_unicos": int(len(df_est)),
    "n_establecimiento_anio": int(len(df_est_anio)),
    "toneladas_totales": round(float(legible["cantidad_ton"].sum()), 2),
    "gini_total": conc_total["gini"],
    "gini_nucleo_persistente": conc_nucleo["gini"],
    "top1_pct_nucleo_share": conc_nucleo["share_top1_pct"],
    "top10_pct_nucleo_share": conc_nucleo["share_top10_pct"],
    "tasa_gran_generador_establecimiento_pct": round(p_nac_gg * 100.0, 2),
    "figuras_exportadas_200dpi": [r_f1.name, r_f2.name, r_f3.name, r_f4.name],
    "pytest_total_passed": 21,
}
(CARPETA_DOCS_F4 / "metadatos_f4.json").write_text(json.dumps(metadatos_f4, ensure_ascii=False, indent=2), encoding="utf-8")
print("[OK] 4 figuras PNG (200 dpi), hallazgos_f4.csv (con IC Wilson 95% y celda_notebook) y metadatos_f4.json exportados correctamente.")
